# STM Metro (Data Profiling)

Profiling the two translated data files

- `data/metro_incidents_en.csv` — one row per incident, 2019 to 2025
- `data/planned_mileage_en.csv` — planned km by line and day, 2019 to 2023

Sections:

1. Load the files
2. Missingness (three different kinds)
3. Categorical levels
4. Service Time Parsing
5. Multi-line incidents 
6. Cleaning the mileage file
7. Merging Data (Fixing dates and Handling the Route variable)
8. Rates, and picking the right denominator
9. Visualization

## 1. Load the files

Read everything as text first. If we let pandas guess the types it turns the `N/A`
strings into nulls, and we want to keep those separate.

In [80]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

INCIDENTS_PATH = 'data/metro_incidents_en.csv'
MILEAGE_PATH = 'data/planned_mileage_en.csv'

# keep_default_na=False stops pandas turning 'N/A' into NaN.
# Only a genuinely empty cell counts as null.
inc = pd.read_csv(INCIDENTS_PATH, dtype=str, keep_default_na=False, na_values=[''])
inc['day'] = pd.to_datetime(inc.calendar_day)

km = pd.read_csv(MILEAGE_PATH, dtype=str, keep_default_na=False, na_values=[''])

print('incidents:', inc.shape)
print('mileage:  ', km.shape)

incidents: (45561, 27)
mileage:   (8472, 11)


In [81]:
inc.columns

Index(['incident_number', 'type_of_incident', 'primary_cause', 'secondary_cause', 'symptom', 'line', 'route', 'time_of_incident', 'resumption_time', 'incident_duration_band', 'vehicle', 'car_door',
       'hardware_type', 'location_code', 'material_damage', 'kfs', 'door', 'emergency_metro', 'cat', 'evacuation', 'calendar_year', 'calendar_year_month', 'calendar_month', 'day_of_the_month',
       'day_of_the_week', 'calendar_day', 'day'],
      dtype='str')

In [82]:
km.head(3)

,line,operating_period,day_type,route,calendar_year,calendar_year_month,calendar_week_start,calendar_day,day_of_the_week,calendar_month,planned_mileage
0,blue,18N,sunday,001,2019,2019-01,2018-12-31,2019-01-06,sunday,1,15810.0
1,blue,18N,fourth holiday period,001,2019,2019-01,2018-12-31,2019-01-02,wednesday,1,15810.0
2,blue,18N,saturday,001,2019,2019-01,2018-12-31,2019-01-05,saturday,1,16315.92


## 2. Missingness

Three different things in these files look like "missing" but mean different things:

| Token | What it means |
|---|---|
| `N/A` | The field doesn't apply. A station incident has no train, so no vehicle number. # in data|
| `unassigned` | The field applies but nobody filled it in. |
| empty cell | A real null — something broke upstream. |

Count them separately instead of lumping them together.

In [83]:
def missing_report(df):
    """Count each kind of missing value, one row per column."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            'column': col,
            'n_unique': s.nunique(),
            'empty': s.isna().sum(),
            'na_string': (s == 'N/A').sum(),
            'unassigned': (s == 'unassigned').sum(),
        })
    out = pd.DataFrame(rows)
    out['total_missing'] = out['empty']  + out['unassigned'] #+ out['na_string']
    out['pct_missing'] = (out['total_missing'] / len(df) * 100).round(1)
    return out.sort_values('pct_missing', ascending=False)


missing_report(inc).head()

,column,n_unique,empty,na_string,unassigned,total_missing,pct_missing
12,hardware_type,3,0,0,41416,41416,90.9
2,primary_cause,5,482,0,0,482,1.1
3,secondary_cause,20,482,0,0,482,1.1
4,symptom,7,377,0,0,377,0.8
5,line,13,0,0,103,103,0.2


In [84]:
missing_report(km).head(3)

,column,n_unique,empty,na_string,unassigned,total_missing,pct_missing
2,day_type,8,0,0,1536,1536,18.1
0,line,4,0,0,0,0,0.0
1,operating_period,28,0,1536,0,0,0.0


### Missingness has to be read per incident type

`hardware_type` looks 91% missing. But station incidents don't involve a train, so the
field is going to always be missing for them. Splitting by `type_of_incident` separates
those two out.

In [85]:
for kind in ['train', 'station']:
    subset = inc[inc.type_of_incident == kind]
    report = missing_report(subset)
    report = report[report.pct_missing > 0]
    print('---', kind, '(', len(subset), 'rows )')
    print(report[['column', 'na_string', 'unassigned', 'empty', 'pct_missing']].to_string(index=False))
    print()

--- train ( 29167 rows )
         column  na_string  unassigned  empty  pct_missing
  hardware_type          0       25022      0         85.8
  primary_cause          0           0    482          1.7
secondary_cause          0           0    482          1.7
        symptom          0           0     83          0.3
           line          0          97      0          0.3

--- station ( 16394 rows )
       column  na_string  unassigned  empty  pct_missing
hardware_type          0       16394      0        100.0
      symptom          0           0    294          1.8



Missingess seems to increase year on year

In [86]:
train = inc[inc.type_of_incident == 'train']

completeness = train.groupby(train.day.dt.year).agg(
    route_na=('route', lambda s: (s == 'N/A').mean() * 100),
    vehicle_na=('vehicle', lambda s: (s == 'N/A').mean() * 100),
    hardware_unassigned=('hardware_type', lambda s: (s == 'unassigned').mean() * 100),
).round(1)
completeness

,route_na,vehicle_na,hardware_unassigned
day,,,
2019,14.7,59.6,81.2
2020,18.4,60.2,81.1
2021,18.0,63.3,85.6
2022,18.9,60.3,85.9
2023,19.7,62.1,87.8
2024,19.7,56.8,90.6
2025,23.0,60.6,88.2


## 3. Categorical levels

In [87]:
for col in ['type_of_incident', 'line', 'symptom', 'primary_cause',
            'hardware_type', 'evacuation', 'incident_duration_band']:
    print('---', col)
    print(inc[col].value_counts(dropna=False).to_string())

--- type_of_incident
type_of_incident
train      29167
station    16394
--- line
line
green                       20047
orange                      17994
blue                         4640
yellow                       2398
green_orange                  164
unassigned                    103
green_orange_yellow           101
orange_blue                    63
green_orange_yellow_blue       28
green_orange_blue              10
orange_yellow                   6
green_yellow                    5
green_blue                      2
--- symptom
symptom
customers                              24397
rolling stock                           7750
station operations                      5176
fixed equipment                         3942
train operations                        2071
fire, smoke, odour, substance, etc.     1712
NaN                                      377
miscellaneous                            136
--- primary_cause
primary_cause
other               18082
customers           15693
fixed eq

Two things stand out:

- `line` has 13 values, not 4. Some incidents affect several lines at once, and 103 are
  `unassigned`.
- `primary_cause` is `other` for a lot of rows. Check whether that's a real category.

In [88]:
# Is 'other' a real cause, or just what station incidents default to?
pd.crosstab(inc.type_of_incident, inc.primary_cause == 'other')

primary_cause,False,True
type_of_incident,,
station,0,16394
train,27479,1688


Almost all `other` rows are station incidents. The documentation says station incidents
don't get a cause analysis, so `primary_cause` and `secondary_cause` are effectively
train-only fields.

## 4. Service Time (>24h) handling

The time columns don't parse with `pd.to_datetime`. The metro service day runs past
midnight, so hours go beyond 24.

In [89]:
# Pull out the hour and check whether it's a valid clock hour.
hour = inc.time_of_incident.str.split(':').str[0].astype(int)

bad = inc[hour >= 24]
print(len(bad), 'start times outside the normal 24h clock')
print(bad.time_of_incident.value_counts().head(8).to_string())

2316 start times outside the normal 24h clock
time_of_incident
24:58:00    53
24:30:00    49
24:20:00    39
24:10:00    38
25:20:00    34
24:04:00    34
24:15:00    34
24:02:00    32


In [90]:
def to_minutes(time_column):
    """Turn 'HH:MM' into minutes since the start of the service day.

    Hours can be 24 or 25 (or higher), meaning after midnight. We leave them
    as-is rather than wrapping, so subtraction still works across midnight.
    """
    parts = time_column.str.split(':', expand=True)
    return parts[0].astype(float) * 60 + parts[1].astype(float)


inc['start_min'] = to_minutes(inc.time_of_incident)
inc['end_min'] = to_minutes(inc.resumption_time)
inc['duration_min'] = inc.end_min - inc.start_min

print('latest start hour: ', inc.start_min.max() / 60)
print('latest end hour:   ', inc.end_min.max() / 60)
print('negative durations:', (inc.duration_min < 0).sum())
print()
print(inc.duration_min.describe().round(1).to_string())

latest start hour:  25.983333333333334
latest end hour:    29.5
negative durations: 0

count    45561.0
mean        25.8
std         60.9
min          0.0
25%          0.0
50%          4.0
75%         30.0
max       1605.0


### Does the duration band match the clock?

In [91]:
BAND_LOWER = {'02 min and under': 0, '03 to 04 min': 3, '05 to 09 min': 5,
              '10 to 14 min': 10, '15 to 19 min': 15, '20 to 24 min': 20,
              '25 to 29 min': 25, '30 min and over': 30}
BAND_UPPER = {'02 min and under': 2, '03 to 04 min': 4, '05 to 09 min': 9,
              '10 to 14 min': 14, '15 to 19 min': 19, '20 to 24 min': 24,
              '25 to 29 min': 29, '30 min and over': np.inf}
BAND_ORDER = list(BAND_LOWER)

inc['band_lower'] = inc.incident_duration_band.map(BAND_LOWER)
inc['band_upper'] = inc.incident_duration_band.map(BAND_UPPER)
inc['band_fits_clock'] = (inc.duration_min >= inc.band_lower) & (inc.duration_min <= inc.band_upper)

for kind in ['train', 'station']:
    subset = inc[inc.type_of_incident == kind]
    summary = subset.groupby('incident_duration_band').agg(
        n=('duration_min', 'size'),
        pct_fits=('band_fits_clock', lambda s: round(s.mean() * 100, 1)),
        median=('duration_min', 'median'),
        p95=('duration_min', lambda s: s.quantile(0.95)),
        max=('duration_min', 'max'),
    ).reindex(BAND_ORDER).dropna(how='all')
    print('---', kind)
    print(summary.round(1).to_string())
    print()

--- train
                            n  pct_fits  median    p95     max
incident_duration_band                                        
02 min and under        16450      98.9     0.0    2.0  1202.0
03 to 04 min             6304      97.5     3.0    4.0   244.0
05 to 09 min             4377      95.3     6.0    9.0   450.0
10 to 14 min              933      93.5    11.0   16.0   326.0
15 to 19 min              299      94.6    17.0   19.0    93.0
20 to 24 min              173      92.5    22.0   25.0    39.0
25 to 29 min              107      93.5    27.0   33.7   113.0
30 min and over           524      99.8    49.0  125.0  1170.0

--- station
                              n  pct_fits  median    p95     max
incident_duration_band                                          
02 min and under        16394.0       4.5    41.0  216.0  1605.0



Every station incident is labeled with `02 min and under` band, but their median real
duration is around 41 minutes. The band only means something for train incidents.

**Don't use `incident_duration_band` as a severity measure for station incidents.**

In [92]:
# The incident number carries a date: 'T' or 'S', then ddmmyy, then a sequence number.
embedded_date = inc.incident_number.str[1:7]
matches = embedded_date == inc.day.dt.strftime('%d%m%y')
mismatched = inc[~matches]
print('of the', len(mismatched), 'mismatches,',
      (mismatched.start_min >= 24 * 60).sum(), 'started after midnight')

of the 2376 mismatches, 2316 started after midnight


In [93]:
train = inc[inc.type_of_incident == 'train']
mismatched = train[~train.band_fits_clock]

print('Example of train incidents where the clock falls outside the band')
train[(train.incident_duration_band == '02 min and under') & (train.duration_min > 120)][[
    'incident_number', 'line', 'symptom', 'time_of_incident','resumption_time', 'duration_min','day']
    ].sort_values('duration_min',ascending=False).head(8)

#"fixed equipment" incident happening at the same time of day

Example of train incidents where the clock falls outside the band


,incident_number,line,symptom,time_of_incident,resumption_time,duration_min,day
30371,T15082502,orange,rolling stock,07:39,27:41:00,1202.0,2025-08-15
32370,T17102402,blue,fixed equipment,05:00,24:30:00,1170.0,2024-10-17
20042,T04102417,blue,fixed equipment,05:30,24:30:00,1140.0,2024-10-04
23858,T08102404,blue,fixed equipment,05:30,24:30:00,1140.0,2024-10-08
21049,T05102411,blue,fixed equipment,05:30,24:30:00,1140.0,2024-10-05
21982,T06102409,blue,fixed equipment,05:30,24:30:00,1140.0,2024-10-06
22911,T07102411,blue,fixed equipment,05:30,24:30:00,1140.0,2024-10-07
22123,T06122317,orange,customers,02:00,18:36,996.0,2023-12-07


In [94]:
# everything above 30 min is a single label 
capped = train[train.incident_duration_band == '30 min and over']
capped.duration_min.describe(percentiles=[.5, .75, .95]).round(1)

count     524.0
mean       67.0
std        87.7
min        20.0
50%        49.0
75%        68.0
95%       125.0
max      1170.0
Name: duration_min, dtype: float64

In [95]:
drift = (~train.band_fits_clock).groupby(train.day.dt.year).mean() * 100
drift.round(2)

day
2019    1.52
2020    1.99
2021    2.13
2022    2.33
2023    2.36
2024    2.53
2025    2.86
Name: band_fits_clock, dtype: float64

### Some incidents are too close to each other

In [96]:
CLUSTER_WINDOW = 10 #minutes 
inc_cluster = inc.sort_values(['line', 'day', 'start_min'])
inc_cluster['mins_since_prev'] = inc_cluster.groupby(['line', 'day'])['start_min'].diff()
inc_cluster['cluster_id'] = (inc_cluster.mins_since_prev.isna() | (inc_cluster.mins_since_prev 
                                                                   >= CLUSTER_WINDOW)).apply(lambda x: int(x)).cumsum()

inc_cluster.groupby('cluster_id').agg({
    'incident_number':'count'}).sort_values(['incident_number'], ascending=False).head(10)


,incident_number
cluster_id,
26365,10
38832,9
31465,6
8174,6
7932,5
11311,5
5218,5
7552,5
32622,5


In [97]:
##Example 
close_incidents =  inc_cluster[inc_cluster['cluster_id']==26365]['incident_number']
inc[inc['incident_number'].isin(close_incidents)][['incident_number',
                                                   'symptom',
                                                   'line', 
                                                   'location_code',
                                                   'incident_duration_band', 
                                                   'day','time_of_incident']]

,incident_number,symptom,line,location_code,incident_duration_band,day,time_of_incident
10401,S19121906,customers,orange,Rosemont,02 min and under,2019-12-19,15:39
34488,T19121909,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:15
34489,T19121910,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:20
34490,T19121911,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:24
34491,T19121912,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:29
34492,T19121913,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:37
34494,T19121915,customers,orange,Rosemont,03 to 04 min,2019-12-19,15:39
34495,T19121916,rolling stock,orange,Côte-Vertu,02 min and under,2019-12-19,15:44
34496,T19121917,train operations,orange,Lionel-Groulx,02 min and under,2019-12-19,15:49
34497,T19121918,train operations,orange,Berri L-2,02 min and under,2019-12-19,15:57


## 5. Multi-line incidents

`line` has 13 values because some incidents hit several lines at once. It's tempting to
drop them — they're under 1% of rows. Check what they look like first.

In [98]:
SINGLE_LINES = ['green', 'orange', 'blue', 'yellow']
MAJOR_BANDS = ['20 to 24 min', '25 to 29 min', '30 min and over']

inc['is_major'] = inc.incident_duration_band.isin(MAJOR_BANDS)

# Label each row: one line, several lines, or no line recorded.
def line_class(value):
    if value in SINGLE_LINES:
        return 'single'
    if value == 'unassigned':
        return 'unassigned'
    return 'multi'

inc['line_class'] = inc.line.apply(line_class)

inc.groupby('line_class').agg(
    n=('incident_number', 'size'),
    pct_major=('is_major', lambda s: round(s.mean() * 100, 2)),
    pct_evacuation=('evacuation', lambda s: round((s != 'N/A').mean() * 100, 2)),
)

,n,pct_major,pct_evacuation
line_class,,,
multi,379,21.64,22.16
single,45079,1.59,2.85
unassigned,103,2.91,1.94


In [99]:
# How much of the severe stuff lives in that 0.8%?
share_of_rows = (inc.line_class == 'multi').mean() * 100
share_of_major = (inc[inc.is_major].line_class == 'multi').mean() * 100

print(f'multi-line incidents are {share_of_rows:.1f}% of all rows')
print(f'  but {share_of_major:.1f}% of all major (20+ min) incidents')

multi-line incidents are 0.8% of all rows
  but 10.2% of all major (20+ min) incidents


Multi-line incidents are 0.8% of rows and 10% of major incidents — a 13x enrichment.
Dropping them removes a tenth of exactly the events we care about, and the tenth with
the strongest signal. An incident spreading across lines pretty much *is* a major
interruption.

So we explode them: one row per affected line, full credit to each. From a rider's
point of view on the green line, a green+orange incident is a green line incident.
We keep `n_lines_affected` so we can switch to fractional credit later if we ever need
to attribute cost without double-counting.

In [100]:
# Split 'green_orange' into ['green', 'orange'] and give each line its own row.
to_explode = inc[inc.line_class != 'unassigned'].copy()
to_explode['affected_line'] = to_explode.line.str.split('_')
to_explode['n_lines_affected'] = to_explode.affected_line.str.len()
inc_by_line = to_explode.explode('affected_line')

print('incidents in:', len(to_explode))
print('rows out:    ', len(inc_by_line))
print()
print(inc_by_line.affected_line.value_counts().to_string())

incidents in: 45458
rows out:     46004

affected_line
green     20357
orange    18366
blue       4743
yellow     2538


In [101]:
# Sanity check
# how much did exploding change each line's total
dropped = inc[inc.line_class == 'single'].line.value_counts()
kept = inc_by_line.affected_line.value_counts()

# Note: don't name a column 'explode' -- it collides with DataFrame.explode()
comparison = pd.DataFrame({'if_dropped': dropped, 'if_exploded': kept})
comparison['pct_change'] = ((comparison.if_exploded / comparison.if_dropped - 1) * 100).round(1)
comparison

,if_dropped,if_exploded,pct_change
green,20047,20357,1.5
orange,17994,18366,2.1
blue,4640,4743,2.2
yellow,2398,2538,5.8


Line totals move by 1 to 6%, so exploding costs very little distortion and buys back
all the severe events. The 103 `unassigned` rows kept out of the line-level analysis (they have an unusually high evacuation rate)

## 6. Cleaning the mileage file

Some days have 8 rows instead of 4.

In [102]:
km['day'] = pd.to_datetime(km.calendar_day)
km['planned_km'] = km.planned_mileage.astype(float)
rows_per_day = km.groupby('day').size()
#Example
print(km[km.day == '2019-01-02'][['line', 'operating_period', 'day_type', 'planned_km']].to_string(index=False))

  line operating_period              day_type  planned_km
  blue              18N fourth holiday period   15810.000
  blue              N/A            unassigned       0.000
yellow              18N fourth holiday period   10045.980
yellow              N/A            unassigned       0.000
orange              18N fourth holiday period   76026.204
orange              N/A            unassigned       0.000
 green              18N fourth holiday period   55780.920
 green              N/A            unassigned       0.000


In [103]:
# The extra rows are zero-mileage padding, so drop them.
padding = km.operating_period.isna()
km = km[~padding].copy()
print('mileage rows kept:', len(km))

mileage rows kept: 8472


## 7. Merging the two datasets

### Date columns

`calendar_year_month` is populated now, but the two files use different formats, so it
can't be used as a join key as-is.

In [104]:
print('incidents:', inc.calendar_year_month.unique()[:3], '| nulls:', inc.calendar_year_month.isna().sum())
print('mileage:  ', km.calendar_year_month.unique()[:3], '| nulls:', km.calendar_year_month.isna().sum())

# Safer to derive month from the date column in both files.
inc['day'] = pd.to_datetime(inc.calendar_day)
inc['month'] = inc.day.dt.to_period('M').dt.to_timestamp()
km['month'] = km.day.dt.to_period('M').dt.to_timestamp()

print()
print('incidents:', inc.day.min().date(), 'to', inc.day.max().date())
print('mileage:  ', km.day.min().date(), 'to', km.day.max().date())

incidents: <StringArray>
['Jan-2019', 'Jan-2020', 'Jan-2021']
Length: 3, dtype: str | nulls: 0
mileage:   <StringArray>
['2019-01', '2019-02', '2019-03']
Length: 3, dtype: str | nulls: 0

incidents: 2019-01-01 to 2025-09-30
mileage:   2019-01-01 to 2023-09-30


### Why can't join on `route`

Both files have a column called `route`, and incident `route`  was modified to a use zero-padded codes like `001` 
to match the `route` code in mileaege during translation in case it was a join key but the checks below demonstrate it isn't. 

In [105]:
# Check 1: how many values does each file actually use?
print('mileage route values:  ', km.route.unique())
print('incident route values: ', inc.route.nunique(), 'distinct')
print('  sample:', sorted(inc.loc[inc.route != 'N/A', 'route'].unique())[:12])
print('  range: ', pd.to_numeric(inc.route, errors='coerce').min(),
      'to', pd.to_numeric(inc.route, errors='coerce').max())

mileage route values:   <StringArray>
['001']
Length: 1, dtype: str
incident route values:  168 distinct
  sample: ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012']
  range:  1.0 to 779.0


The mileage file uses exactly one code. The incidents file uses 168, spanning 001 to
779. Those look like individual train run numbers.

The obvious worry: maybe STM only gave us mileage for one route out of many, and we'd
be normalising incidents from the whole network by the mileage of a single train.
Check 2 rules that out.

In [106]:
# Check 2: is route '001' one train run, or the whole line's service?
# If it were a single train, the daily mileage would be small.

## Obtained from Wikipedia
LINE_LENGTH_KM = {'green': 22.1, 'orange': 30.0, 'blue': 9.7, 'yellow': 4.25}

daily = km.groupby(['line', 'day']).planned_km.sum().groupby('line').median()

check = pd.DataFrame({'median_km_per_day': daily.round(0)})
check['line_length_km'] = pd.Series(LINE_LENGTH_KM)
check['one_way_trips_per_day'] = (check.median_km_per_day / check.line_length_km).round(0)
check

,median_km_per_day,line_length_km,one_way_trips_per_day
line,,,
blue,23968.0,9.70,2471.0
green,93443.0,22.10,4228.0
orange,127923.0,30.00,4264.0
yellow,13338.0,4.25,3138.0


No single train makes 2,000+ one-way trips a day.
The annual total confirms it — around 85 to 90 million car-km, which matches the
figure STM publishes for the entire network. One train run would be roughly 0.5 million. So `route='001'` in the mileage file means **all services**, not one run. The two columns share a name but mean different things

A join on `route` keeps about 2% of incidents, and the ones it keeps get matched to the
whole network's mileage rather than their own.

**Conclusion: join on line + day. Ignore `route`.**

### Merging on line + day

In [107]:
km.shape

(8472, 14)

In [108]:
# Mileage: one row per line per day
mileage_daily = km.groupby(['line', 'day'], as_index=False).planned_km.sum()

print(mileage_daily.shape)

# Incidents: same grain, using the exploded frame
incidents_daily = inc_by_line.groupby(['affected_line', 'day'], as_index=False).agg(
    n_incidents=('incident_number', 'size'),
    n_major=('is_major', 'sum'),
    total_delay_min=('duration_min', 'sum'),
).rename(columns={'affected_line': 'line'})

print(incidents_daily.shape)

(6936, 3)
(8250, 5)


In [109]:
merged = mileage_daily.merge(incidents_daily, on=['line', 'day'],how='outer', indicator=True)
merged._merge.value_counts()

_merge
both          5763
right_only    2487
left_only     1173
Name: count, dtype: int64

In [110]:
# left_only  = line ran, no incidents that day ->  fill with zeros
usable = merged[merged._merge != 'right_only'].copy()
for col in ['n_incidents', 'n_major', 'total_delay_min']:
    usable[col] = usable[col].fillna(0)

# right_only = incidents after Sept 2023 -> no mileage available
no_denominator = merged[merged._merge == 'right_only']

print('usable line-days:', len(usable), ' covering',
      usable.day.min().date(), 'to', usable.day.max().date())
print('incidents with no mileage:', int(no_denominator.n_incidents.sum()))

usable line-days: 6936  covering 2019-01-01 to 2023-09-30
incidents with no mileage: 14949


## 8. Rates, and picking the right denominator

Dividing by planned mileage fixes the ranking problem. But planned mileage counts
*trains*, and a lot of incidents are caused by *passengers*, who don't scale with train
kilometres. So it matters which incidents we normalise.

In [111]:
usable['year'] = usable.day.dt.year

yearly = usable.groupby(['year', 'line'], as_index=False).agg(
    incidents=('n_incidents', 'sum'),
    planned_km=('planned_km', 'sum'),
)
print('RAW COUNTS')
yearly.pivot(index='year', columns='line', values='incidents').astype(int)

RAW COUNTS


line,blue,green,orange,yellow
year,,,,
2019,845,3340,3384,447
2020,633,2486,2325,239
2021,554,2466,2249,245
2022,694,3003,2533,363
2023,500,2538,1943,268


In [112]:
yearly['per_100k_km'] = (yearly.incidents / yearly.planned_km * 1e5).round(2)
print('PER 100,000 PLANNED KM')
print(yearly.pivot(index='year', columns='line', values='per_100k_km').to_string())

PER 100,000 PLANNED KM
line   blue  green  orange  yellow
year                              
2019  10.01  10.26    7.69    9.13
2020   7.62   7.73    5.44    5.03
2021   7.04   8.01    5.47    5.38
2022   8.82   9.75    6.16    7.96
2023   8.70  11.85    6.51    5.89


Orange has high (most or second most) incidents in all yeats but has lower (lowest or second lowest) rates. 
Ranking by raw count would point resources at the busiest line rather than the least reliable one.

But there are limitations. See below 

Limitation 1. This is *planned* mileage not actual mileage.

### The COVID-19 effect

In [113]:
monthly = usable.groupby('month' if 'month' in usable else usable.day.dt.to_period('M').dt.to_timestamp()).agg(
    planned_km=('planned_km', 'sum'), incidents=('n_incidents', 'sum'))
monthly.index.name = 'month'

before = monthly.loc['2019-10':'2020-02'].mean() # 5 months before covid
during = monthly.loc['2020-04':'2020-08'].mean() # 5 months during covid

print('Pre-COVID\n',before.to_string(), '\n\nDuring COVID\n', during.to_string())


Pre-COVID
 planned_km    7.605554e+06
incidents     7.146000e+02 

During COVID
 planned_km    7.364768e+06
incidents     3.982000e+02


In [114]:
print('service cut: ', round((1 - during.planned_km / before.planned_km) * 100), '%')
print('incidents fell: ', round((1 - during.incidents / before.incidents) * 100), '%')

service cut:  3 %
incidents fell:  44 %


Service barely changed. Incidents fell 44%. The denominator is depending on planned mileage so it is unaffected by 
a major disruption of a pandemic, because it records what STM*planned* to run, not what actually ran or how many people rode.

Limitation 2. Passenger incidents don't scale with train kilometres at all.

In [115]:
inc.columns

Index(['incident_number', 'type_of_incident', 'primary_cause', 'secondary_cause', 'symptom', 'line', 'route', 'time_of_incident', 'resumption_time', 'incident_duration_band', 'vehicle', 'car_door',
       'hardware_type', 'location_code', 'material_damage', 'kfs', 'door', 'emergency_metro', 'cat', 'evacuation', 'calendar_year', 'calendar_year_month', 'calendar_month', 'day_of_the_month',
       'day_of_the_week', 'calendar_day', 'day', 'start_min', 'end_min', 'duration_min', 'band_lower', 'band_upper', 'band_fits_clock', 'is_major', 'line_class', 'month'],
      dtype='str')

In [116]:
# Incidents per day, weekday vs weekend

inc['is_weekend'] = inc.day.dt.dayofweek >= 5
days = inc.groupby('is_weekend').day.nunique()
# stratified by what caused them
per_day = inc.groupby(['symptom', 'is_weekend']).size().unstack()

## avg incidents per day
per_day = (per_day / days).round(2)
##rename columns
per_day.columns = ['weekday', 'weekend']
per_day['weekend_vs_weekday'] = (per_day.weekend / per_day.weekday).round(2)

per_day.sort_values('weekend_vs_weekday', ascending=False)

,weekday,weekend,weekend_vs_weekday
symptom,,,
station operations,2.08,2.15,1.03
customers,10.05,9.51,0.95
"fire, smoke, odour, substance, etc.",0.77,0.49,0.64
fixed equipment,1.79,1.12,0.63
rolling stock,3.58,2.05,0.57
train operations,0.97,0.52,0.54
miscellaneous,0.07,0.03,0.43


Train faults roughly halve on weekends, tracking reduced service. Passenger incidents
barely move. So on weekends the denominator drops while the numerator doesn't,
which makes weekends look less reliable than they are.

**Deicision: Maybe Split the denominators**

- Rolling stock / train operations / fixed equipment → per 100k planned km
- Customers / station operations → count per day, as we have no ridership data

In [117]:
TRAIN_SYMPTOMS = ['rolling stock', 'train operations', 'fixed equipment']

train_related = inc_by_line[inc_by_line.symptom.isin(TRAIN_SYMPTOMS)]
train_daily = train_related.groupby(['affected_line', 'day']).size().rename('n_train_related')
train_daily = train_daily.reset_index().rename(columns={'affected_line': 'line'})

## mileage_daily defined in 7 (Merging on line + day)
split = mileage_daily.merge(train_daily,on=['line', 'day'], how='left')
split['n_train_related'] = split.n_train_related.fillna(0)
split['year'] = split.day.dt.year

inc_by_year = split.groupby(['year', 'line'], as_index=False).agg(
    n=('n_train_related', 'sum'), planned_km=('planned_km', 'sum'))
inc_by_year['per_100k_km'] = (inc_by_year.n / inc_by_year.planned_km * 1e5).round(2)

print('TRAIN-RELATED INCIDENTS PER 100,000 PLANNED KM')
inc_by_year.pivot(index='year', columns='line', values='per_100k_km')

TRAIN-RELATED INCIDENTS PER 100,000 PLANNED KM


line,blue,green,orange,yellow
year,,,,
2019,4.76,3.28,2.22,5.72
2020,3.32,2.42,1.65,3.24
2021,2.80,2.06,1.59,3.16
2022,3.89,2.28,1.77,4.50
2023,3.81,2.74,1.79,3.69


## 9. Data Visualization

In [118]:
LINE_COLOURS = {'green': '#00a54f', 'orange': '#f68b1f',
                'blue': '#0072ce', 'yellow': '#f2c500'}

px.defaults.color_discrete_map = LINE_COLOURS
px.defaults.template = 'plotly_white'
px.defaults.width, px.defaults.height = 1000, 400

In [119]:
# Raw monthly counts
inc_by_line['month'] = inc_by_line.day.dt.to_period('M').dt.to_timestamp()
counts = inc_by_line.groupby(['month', 'affected_line']).size().reset_index(name='incidents')

px.line(counts, x='month', y='incidents', color='affected_line',
        title='Raw monthly incidents')

In [120]:
# Same thing normalised by planned km
usable['month'] = usable.day.dt.to_period('M').dt.to_timestamp()
rates = usable.groupby(['month', 'line'], as_index=False).agg(
    incidents=('n_incidents', 'sum'), planned_km=('planned_km', 'sum'))
rates['per_100k_km'] = rates.incidents / rates.planned_km * 1e5

px.line(rates, x='month', y='per_100k_km', color='line',
        title='Per 100k planned km — orange is the most reliable, not the worst')

In [121]:
# Severity of incident distribution, and how station rows all pile into one bucket
# The duration band was used as a proxy/measure for severity
BAND_ORDER = ['02 min and under', '03 to 04 min', '05 to 09 min', '10 to 14 min',
              '15 to 19 min', '20 to 24 min', '25 to 29 min', '30 min and over']

px.histogram(inc, y='incident_duration_band', color='type_of_incident',
             category_orders={'incident_duration_band': BAND_ORDER[::-1]},
             color_discrete_map={'train': '#c0392b', 'station': '#95a5a6'},
             title='Incidents (All Station incident labeled "02 min and under")')

##### $$\text{Lift} = \frac{P(\text{Long Incident} \mid \text{Flag Present})}{P(\text{Long Incident Overall})}$$

In [122]:
# Which flags, known when the incident opens, predict a long one?
## Serves as a prior in the data


base_rate = (inc.incident_duration_band == '30 min and over').mean()

flags = pd.DataFrame({
    'emergency_metro': inc.emergency_metro == 'True',
    'material_damage': inc.material_damage == 'True',
    'cat': inc.cat == 'True',
    'kfs': inc.kfs == 'True',
    'door': inc.door == 'True',
    'evacuation': inc.evacuation != 'N/A',
})
#print(flags.shape[0] == inc.shape[0])

is_long = inc.incident_duration_band == '30 min and over'

lift = pd.DataFrame({
    'n': flags.sum(),
    'pct_30min': (flags.apply(lambda f: is_long[f].mean(), axis=0) * 100).round(2) ## calculates the cinditional prob.,
})


lift['lift'] = (lift.pct_30min / 100 / base_rate).round(1)
lift = lift.sort_values('lift').reset_index(names='flag')
print(lift)
print('base rate of 30min+ incidents:', round(base_rate * 100, 2), '%')
px.bar(lift, x='lift', y='flag', orientation='h', text='n', log_x=True,
       color=lift.lift > 1, color_discrete_map={True: '#c0392b', False: '#7f8c8d'},
       title='Lift on P(incident runs 30+ min)>1. Bar labels are sample size')

              flag     n  pct_30min  lift
0             door  2911       0.10   0.1
1              kfs  1054       0.76   0.7
2              cat  7045       6.52   5.7
3  material_damage   353      28.05  24.4
4       evacuation  1371      29.83  25.9
5  emergency_metro   479      44.47  38.7
base rate of 30min+ incidents: 1.15 %


In [123]:
usable[['planned_km','n_incidents','line','day']].head()

,planned_km,n_incidents,line,day
0,14039.28,1.0,blue,2019-01-01
1,15810.00,1.0,blue,2019-01-02
2,26940.24,1.0,blue,2019-01-03
3,26940.24,2.0,blue,2019-01-04
4,16315.92,3.0,blue,2019-01-05


In [124]:
# The pooled correlation looks strong but disappears inside each line
global_corr = usable.planned_km.corr(usable.n_incidents)
within = usable.groupby('line').apply(lambda g: g.planned_km.corr(g.n_incidents)).reset_index(name='correlation')
within['r_squared'] = within['correlation']**2
print(within)
print('Global correlation across all lines:', round(global_corr, 2))
px.bar(within, x='line', y='correlation', color='line',
       title=f'Pearson Correlation within each line (Global Corr = {global_corr:.2f})')

##Only 2-7% of variation is explained

     line  correlation  r_squared
0    blue     0.172942   0.029909
1   green     0.204210   0.041702
2  orange     0.266335   0.070934
3  yellow     0.135553   0.018375
Global correlation across all lines: 0.71


In [125]:
# Does running more service on a given day mean more incidents that day?
px.scatter(usable, x='planned_km', y='n_incidents', color='line', opacity=0.3,
           facet_col='line', facet_col_wrap=2, height=650,
           title='Within each line, more scheduled km does not mean more incidents')

In [126]:
# When do incidents happen, and when are they worst?
inc['hour'] = (inc.start_min // 60).astype(int)
by_hour = inc.groupby('hour').agg(
    incidents=('incident_number', 'size'),
    pct_major=('is_major', lambda s: s.mean() * 100)).reset_index()

px.bar(by_hour, x='hour', y='incidents', title='Volume peaks at rush hour')

In [127]:
px.line(by_hour, x='hour', y='pct_major', markers=True,
        title='But severity peaks late at night, when recovery options are thin',
        labels={'pct_major': '% running 20+ min', 'hour': 'hour of service day (24, 25 = after midnight)'})

In [128]:
# Where do the major incidents cluster?
top_locations = (inc[inc.is_major].location_code.value_counts()
                 .head(25).rename_axis('location').reset_index(name='incidents'))

px.bar(top_locations, x='incidents', y='location', orientation='h', height=600,
       title='Locations with the most major incidents')